# Fake News Detection Model
This notebook implements a machine learning model to classify whether a news article is fake or true.

## 1. Import Required Libraries

In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 2. Load and Explore Data

In [2]:
# Load fake and true news datasets
fake = pd.read_csv("../Data/Fake.csv", engine="python", on_bad_lines="skip")
true = pd.read_csv("../Data/True.csv", engine="python", on_bad_lines="skip")

print(f"Fake news shape: {fake.shape}")
print(f"True news shape: {true.shape}")

Fake news shape: (23481, 4)
True news shape: (21417, 4)


In [3]:
# Display first few rows
print("Fake News Sample:")
print(fake.head())
print("\nTrue News Sample:")
print(true.head())

Fake News Sample:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  
0  December 31, 2017  
1  December 31, 2017  
2  December 30, 2017  
3  December 29, 2017  
4  December 25, 2017  

True News Sample:
                                               title  \
0  As U.S. budget fight looms, Republicans flip

In [4]:
# Check for missing values
print(f"Fake news missing values:\n{fake.isnull().sum()}")
print(f"\nTrue news missing values:\n{true.isnull().sum()}")

Fake news missing values:
title      0
text       0
subject    0
date       0
dtype: int64

True news missing values:
title      0
text       0
subject    0
date       0
dtype: int64


## 3. Prepare Data

In [5]:
# Add labels: 0 for fake, 1 for true
fake['label'] = 0
true['label'] = 1

print(f"Fake label distribution:\n{fake['label'].value_counts()}")
print(f"\nTrue label distribution:\n{true['label'].value_counts()}")

Fake label distribution:
label
0    23481
Name: count, dtype: int64

True label distribution:
label
1    21417
Name: count, dtype: int64


In [6]:
# Concatenate both datasets
df = pd.concat([fake, true], axis=0)

print(f"Combined dataset shape: {df.shape}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")

Combined dataset shape: (44898, 5)

Label distribution:
label
0    23481
1    21417
Name: count, dtype: int64


In [7]:
# Shuffle the data
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Dataset shuffled successfully")
print(f"Dataset shape: {df.shape}")

Dataset shuffled successfully
Dataset shape: (44898, 5)


In [8]:
# Select only text and label columns
df = df[["text", "label"]]

print(f"Final dataset shape: {df.shape}")
print(f"\nFirst few rows:\n{df.head()}")

Final dataset shape: (44898, 2)

First few rows:
                                                text  label
0  21st Century Wire says Ben Stein, reputable pr...      0
1  WASHINGTON (Reuters) - U.S. President Donald T...      1
2  (Reuters) - Puerto Rico Governor Ricardo Rosse...      1
3  On Monday, Donald Trump once again embarrassed...      0
4  GLASGOW, Scotland (Reuters) - Most U.S. presid...      1


## 4. Text Preprocessing

In [9]:
def clean(text):
    """
    Clean text by converting to lowercase and removing special characters
    """
    text = text.lower()
    text = re.sub(r"[^a-zA-Z ]", "", text)
    return text

# Test the cleaning function
sample_text = "Hello! This is a TEST. #FakeNews @User123"
print(f"Original: {sample_text}")
print(f"Cleaned: {clean(sample_text)}")

Original: Hello! This is a TEST. #FakeNews @User123
Cleaned: hello this is a test fakenews user


In [10]:
# Apply cleaning to all text
df["text"] = df["text"].apply(clean)

print("Text cleaning completed")
print(f"Sample cleaned text:\n{df['text'].iloc[0][:200]}")

Text cleaning completed
Sample cleaned text:
st century wire says ben stein reputable professor from pepperdine university also of some hollywood fame appearing in tv shows and films such as ferris bueller s day off made some provocative stateme


## 5. Feature Extraction

In [11]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df["text"], 
    df["label"],
    test_size=0.2,
    random_state=42
)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"\nTraining set label distribution:\n{y_train.value_counts()}")

Training set size: 35918
Test set size: 8980

Training set label distribution:
label
0    18771
1    17147
Name: count, dtype: int64


In [12]:
# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print(f"Training TF-IDF shape: {X_train_tfidf.shape}")
print(f"Test TF-IDF shape: {X_test_tfidf.shape}")

Training TF-IDF shape: (35918, 5000)
Test TF-IDF shape: (8980, 5000)


## 6. Model Training

In [13]:
# Train Logistic Regression model
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_tfidf, y_train)

print("Model training completed!")

Model training completed!


## 7. Model Evaluation

In [14]:
# Make predictions
y_pred = model.predict(X_test_tfidf)

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Model Performance Metrics:")
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

Model Performance Metrics:
Accuracy:  0.9863
Precision: 0.9814
Recall:    0.9899
F1-Score:  0.9857


In [20]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(f"\nConfusion Matrix:\n{cm}")



Confusion Matrix:
[[4630   80]
 [  43 4227]]


## 8. Test Predictions

In [16]:
# Function to predict fake/true news
def predict_news(text):
    """
    Predict whether a news article is fake (0) or true (1)
    """
    cleaned_text = clean(text)
    text_tfidf = vectorizer.transform([cleaned_text])
    prediction = model.predict(text_tfidf)[0]
    probability = model.predict_proba(text_tfidf)[0]
    
    label = "TRUE" if prediction == 1 else "FAKE"
    confidence = probability[prediction] * 100
    
    return label, confidence, probability

print("Prediction function created successfully!")

Prediction function created successfully!


In [17]:
# Test with sample news articles
test_articles = [
    "Donald Trump just couldn't wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and the very dishonest fake news media.",
    "Scientists discover new breakthrough in cancer research that could save millions of lives.",
    "Breaking: Major corporate scandal revealed after months of investigation."
]

print("Testing predictions on sample articles:\n")
for i, article in enumerate(test_articles, 1):
    label, confidence, prob = predict_news(article)
    print(f"Article {i}:")
    print(f"Text: {article[:100]}...")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2f}%")
    print(f"Probabilities - Fake: {prob[0]:.4f}, True: {prob[1]:.4f}")
    print("-" * 80)

Testing predictions on sample articles:

Article 1:
Text: Donald Trump just couldn't wish all Americans a Happy New Year and leave it at that. Instead, he had...
Prediction: FAKE
Confidence: 97.83%
Probabilities - Fake: 0.9783, True: 0.0217
--------------------------------------------------------------------------------
Article 2:
Text: Scientists discover new breakthrough in cancer research that could save millions of lives....
Prediction: FAKE
Confidence: 94.38%
Probabilities - Fake: 0.9438, True: 0.0562
--------------------------------------------------------------------------------
Article 3:
Text: Breaking: Major corporate scandal revealed after months of investigation....
Prediction: FAKE
Confidence: 92.89%
Probabilities - Fake: 0.9289, True: 0.0711
--------------------------------------------------------------------------------


## 9. Save the Model

In [18]:
import pickle

# Save the model
with open('news_detection_model.pkl', 'wb') as f:
    pickle.dump(model, f)

# Save the vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!


In [19]:
# Load the model (for future use)
# with open('news_detection_model.pkl', 'rb') as f:
#     loaded_model = pickle.load(f)
# 
# with open('tfidf_vectorizer.pkl', 'rb') as f:
#     loaded_vectorizer = pickle.load(f)

print("Model loading code ready for future use!")

Model loading code ready for future use!
